Connected to .venv01 (3.13.11) (Python 3.13.11)

In [ ]:
# ANALYSIS TSM
# SEARCH WORDS :
# YEARLY_TSM_ESTIMATION
# TSM_FITTED_TARGET_SPACE_PERFORMANCE
# TSM_SCORE_CALIBRATION
# TSM_THRESHOLD_CALIBRATION
# TSM_THRESHOLD_TRADER_CALIBRATION

import pandas as pd
import numpy as np
import warnings
import os, sys
import pickle

from pathlib import Path
from arch.utility.exceptions import ConvergenceWarning


warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)


sys.path.insert(0,"/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")
sys.path.insert(0,"/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/MFIV_calibration")

os.chdir("/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")

sys.modules.pop("srcB", None)

In [ ]:
# =============================================================================================================
# ======================================= LOAD BASE OUTPUTS ===================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year/outputs_by_year_04.pkl","rb") as f:
    outputs_by_year = pickle.load(f)

In [ ]:
# =============================================================================================================
# ======================================= IMPORTS FROM SRC ====================================================
# =============================================================================================================

from srcB.market import *
from srcB.helpers import *

In [ ]:
# =============================================================================================================
# ======================================= READ D_BTC AND DF ===================================================
# =============================================================================================================

DATA_DIR = Path("../data/rawData")

YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

d_btc_raw_by_year = {}
df_raw_by_year = {}

for y in YEARS:
    d_btc_raw_by_year[y] = pd.read_parquet(DATA_DIR/f"btc_underlying_minutely_{y}.parquet")
    df_raw_by_year[y] = pd.read_parquet(DATA_DIR/f"derivative_data_{y}.parquet")

In [ ]:
# =============================================================================================================
# ======================================= SET INDEX TO DATETIME ===============================================
# =============================================================================================================

for y in YEARS:
    d_btc_raw_by_year[y] = d_btc_raw_by_year[y].set_index(pd.to_datetime(d_btc_raw_by_year[y]["date_time"]))
    df_raw_by_year[y] = df_raw_by_year[y].set_index(pd.to_datetime(df_raw_by_year[y]["date_time"]))

In [ ]:
# =============================================================================================================
# ======================================= LOAD X_BY_YEAR ======================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/x_by_year_fut/x_by_year.pkl", "rb") as f:
    X_by_year = pd.read_pickle(f)

for y in YEARS:
    X_by_year[y] = X_by_year[y].copy()
    X_by_year[y]["rv"] = X_by_year[y]["rt"] ** 2

Xq_by_year = {}

for y in YEARS:
    Xq_by_year[y] = {}
    Xq_by_year[y]["q1"] = X_by_year[y].loc[f"{y}-01-01" : f"{y}-03-31 23:59:59"]
    Xq_by_year[y]["q2"] = X_by_year[y].loc[f"{y}-04-01" : f"{y}-06-30 23:59:59"]
    Xq_by_year[y]["q3"] = X_by_year[y].loc[f"{y}-07-01" : f"{y}-09-30 23:59:59"]
    Xq_by_year[y]["q4"] = X_by_year[y].loc[f"{y}-10-01" : f"{y}-12-31 23:59:59"]

In [ ]:
# =============================================================================================================
# ======================================= MFIV SCALING ========================================================
# =============================================================================================================
# CONVERTS ANNUALIZED MFIV -> CUMULATIVE H-HOUR DECIMAL VARIANCE

MFIVS = list(range(2, 577))

for y in YEARS:
    for q in QUARTERS:
        for H in MFIVS:
            Xq_by_year[y][q][f"MFIV_{H}"] *= (H / 8760.0)

for y in YEARS:
    for q in QUARTERS:
        Xq_by_year[y][q] = Xq_by_year[y][q].loc[:, ~Xq_by_year[y][q].columns.duplicated(keep="first")]

In [ ]:
# =============================================================================================================
# ======================================= BORROW RFR COLUMN ===================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/ready_data/d_btc_by_year_all.pkl", "rb") as f:
    d_btc_by_year = pickle.load(f)

for y in YEARS:
    d_btc_raw_by_year[y]["rfr"] = d_btc_by_year[y]["rfr"].reindex(
        d_btc_raw_by_year[y].index,
        method="ffill"
    )

In [ ]:
# =============================================================================================================
# ======================================= TSM PARAMETERS ======================================================
# =============================================================================================================

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/hyperparameters/tsm_best_params.pkl", "rb") as f:
    best_tsm_params = pickle.load(f)

VAL_FRACTION = 0.20
EPOCHS = 40
PATIENCE = 6
SEED = 42

In [ ]:
# =============================================================================================================
# ======================================= YEARLY_TSM_ESTIMATION ===============================================
# =============================================================================================================

sys.modules.pop("srcB.tsm_model", None)
from srcB.tsm_model import *

tsm_bundle_by_year = {}
tsm_targets_by_year = {}
tsm_output_by_year = {}

for y in YEARS:
    X_df = pd.concat([Xq_by_year[y][q] for q in QUARTERS]).sort_index().copy()
    garch_df = pd.concat([outputs_by_year[y][q]["garch"] for q in QUARTERS]).sort_index()
    X_df["garch"] = garch_df.reindex(X_df.index)

    tsm_output, tsm_bundle, tsm_targets = tsm_mfiv(
        X_df=X_df,
        option_df=df_raw_by_year[y],
        time_df=d_btc_raw_by_year[y],
        sequence_length=168,
        forward_hours=24,
        delta_target=0.20,
        val_fraction=VAL_FRACTION,
        num_blocks=best_tsm_params["num_blocks"],
        feature_hidden=best_tsm_params["feature_hidden"],
        dropout=best_tsm_params["dropout"],
        epochs=EPOCHS,
        batch_size=best_tsm_params["batch_size"],
        learning_rate=best_tsm_params["learning_rate"],
        weight_decay=best_tsm_params["weight_decay"],
        patience=PATIENCE,
        seed=SEED
    )
    tsm_output_by_year[y] = tsm_output
    tsm_bundle_by_year[y] = tsm_bundle
    tsm_targets_by_year[y] = tsm_targets

    for q in QUARTERS:
        q_index = Xq_by_year[y][q].index
        tsm_q = tsm_output.reindex(q_index)

        outputs_by_year[y][q] = outputs_by_year[y][q].drop(
            columns=tsm_q.columns,
            errors="ignore"
        )

        outputs_by_year[y][q] = pd.concat(
            [outputs_by_year[y][q], tsm_q],
            axis=1
        )

In [ ]:
# =============================================================================================================
# ======================================= SAVE TSM BUNDLES ====================================================
# =============================================================================================================

os.makedirs("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/tsm", exist_ok=True)

with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/tsm/tsm_bundle_by_year.pkl", "wb") as f:
    pickle.dump(tsm_bundle_by_year, f)

In [ ]:
# =============================================================================================================
# ================================ TSM_FITTED_TARGET_SPACE_PERFORMANCE ========================================
# =============================================================================================================

target_cols = [
    "target_long_straddle",
    "target_short_straddle",
    "target_long_strangle",
    "target_short_strangle"
]

rows = []

for y in YEARS:
    test = pd.concat([
        tsm_output_by_year[y][["tsm_action"]],
        tsm_targets_by_year[y][target_cols]
    ], axis=1, sort=False).dropna()

    realized = pd.Series(0.0, index=test.index)

    for action in ["long_straddle", "short_straddle", "long_strangle", "short_strangle"]:
        mask = test["tsm_action"] == action
        realized.loc[mask] = test.loc[mask, f"target_{action}"]

    sharpe = realized.mean() / realized.std()

    rows.append({
        "year": y,
        "n": len(realized),
        "mean_target": realized.mean(),
        "target_sharpe": sharpe,
        "profitable": (realized > 0).mean()
    })

target_sharpe = pd.DataFrame(rows)

print(target_sharpe)

   year     n  mean_target  target_sharpe  profitable
0  2019  2194     0.002682       0.092017    0.451231
1  2020  4093     0.002022       0.069033    0.318837
2  2021  5008     0.000640       0.023946    0.606629


In [ ]:
# =============================================================================================================
# ======================================= TSM_SCORE_CALIBRATION ===============================================
# =============================================================================================================

rows = []

for y in YEARS:
    test = pd.concat([
        tsm_output_by_year[y][["tsm_action", "tsm_best_score"]],
        tsm_targets_by_year[y][target_cols]
    ], axis=1, sort=False).dropna()

    test = test[test["tsm_action"] != "no_trade"].copy()
    test["target_pnl"] = 0.0

    for action in ["long_straddle", "short_straddle", "long_strangle", "short_strangle"]:
        m = test["tsm_action"] == action
        test.loc[m, "target_pnl"] = test.loc[m, f"target_{action}"]

    test["score_bin"] = pd.qcut(
        test["tsm_best_score"],
        5,
        labels=False,
        duplicates="drop"
    )

    x = (
        test.groupby("score_bin")
        .agg(
            n=("target_pnl", "size"),
            mean_score=("tsm_best_score", "mean"),
            mean_target=("target_pnl", "mean"),
            sharpe=("target_pnl", lambda z: z.mean() / z.std()),
            profitable=("target_pnl", lambda z: (z > 0).mean())
        )
        .reset_index()
    )

    x["year"] = y
    rows.append(x)

score_check = pd.concat(rows, ignore_index=True)

print(score_check[[
    "year",
    "score_bin",
    "n",
    "mean_score",
    "mean_target",
    "sharpe",
    "profitable"
]].to_string(index=False))

 year  score_bin   n  mean_score  mean_target    sharpe  profitable
 2019          0 358    0.000807    -0.002005 -0.063362    0.438547
 2019          1 357    0.002421    -0.000155 -0.005771    0.507003
 2019          2 358    0.004155     0.003717  0.112004    0.572626
 2019          3 357    0.006393     0.011011  0.297578    0.675070
 2019          4 358    0.010815     0.003897  0.128745    0.575419
 2020          0 558    0.000284     0.001663  0.068826    0.467742
 2020          1 558    0.000940     0.000638  0.027782    0.462366
 2020          2 558    0.001722    -0.000269 -0.011697    0.455197
 2020          3 558    0.002779     0.001663  0.056478    0.464158
 2020          4 558    0.006653     0.011134  0.183177    0.489247
 2021          0 968    0.000858     0.000012  0.000538    0.642562
 2021          1 967    0.001982     0.000797  0.031891    0.628749
 2021          2 968    0.002984     0.001022  0.042241    0.630165
 2021          3 967    0.004141     0.001765  0

In [ ]:
# =============================================================================================================
# ======================================= TSM_THRESHOLD_CALIBRATION ===========================================
# =============================================================================================================

tests = {}

for y in YEARS:
    x = pd.concat([
        tsm_output_by_year[y][["tsm_action", "tsm_best_score"]],
        tsm_targets_by_year[y][target_cols]
    ], axis=1, sort=False).dropna()

    x = x[x["tsm_action"] != "no_trade"].copy()
    x["target_pnl"] = 0.0

    for action in ["long_straddle", "short_straddle", "long_strangle", "short_strangle"]:
        m = x["tsm_action"] == action
        x.loc[m, "target_pnl"] = x.loc[m, f"target_{action}"]

    tests[y] = x


scores_all = pd.concat([
    tests[y]["tsm_best_score"]
    for y in YEARS
])


rows = []

for q in [0, .20, .40, .50, .60, .70, .80, .85, .90]:

    threshold = scores_all.quantile(q)

    for y in YEARS:
        x = tests[y][tests[y]["tsm_best_score"] >= threshold]

        sharpe = x["target_pnl"].mean() / x["target_pnl"].std()

        rows.append({
            "quantile": q,
            "threshold": threshold,
            "year": y,
            "n": len(x),
            "mean_target": x["target_pnl"].mean(),
            "sharpe": sharpe
        })

threshold_check = pd.DataFrame(rows)

print(threshold_check.to_string(index=False))

 quantile  threshold  year    n  mean_target    sharpe
     0.00   0.000002  2019 1788     0.003291  0.102023
     0.00   0.000002  2020 2790     0.002966  0.083702
     0.00   0.000002  2021 4838     0.000663  0.024363
     0.20   0.001132  2019 1541     0.004283  0.130681
     0.20   0.001132  2020 1813     0.003779  0.093483
     0.20   0.001132  2021 4179     0.000829  0.029690
     0.40   0.002157  2019 1309     0.005168  0.157598
     0.40   0.002157  2020 1125     0.006374  0.133245
     0.40   0.002157  2021 3216     0.000765  0.026338
     0.50   0.002719  2019 1181     0.005785  0.175477
     0.50   0.002719  2020  858     0.007976  0.151294
     0.50   0.002719  2021 2669     0.000817  0.027462
     0.60   0.003315  2019 1050     0.006433  0.190951
     0.60   0.003315  2020  609     0.010466  0.178612
     0.60   0.003315  2021 2108     0.000872  0.028008
     0.70   0.004024  2019  912     0.007089  0.201033
     0.70   0.004024  2020  415     0.014775  0.214385
     0.70 

In [ ]:
# =============================================================================================================
# ======================================= TSM SCORE DISTRIBUTION ==============================================
# =============================================================================================================

for y in YEARS:
    x = tests[y]["tsm_best_score"]

    print(
        y,
        "n:", len(x),
        "min:", x.min(),
        "q25:", x.quantile(.25),
        "q50:", x.quantile(.50),
        "q75:", x.quantile(.75),
        "q90:", x.quantile(.90),
        "max:", x.max()
    )

2019 n: 1788 min: 2.4200417e-06 q25: 0.0020202064188197255 q50: 0.004109099740162492 q75: 0.007003776496276259 q90: 0.010069852694869041 max: 0.02935844
2020 n: 2790 min: 2.2590393e-06 q25: 0.0007452946738339961 q50: 0.0017103624995797873 q75: 0.0030653050635010004 q90: 0.004852530546486378 max: 0.03566539
2021 n: 4838 min: 3.2231328e-06 q25: 0.0017355832096654922 q50: 0.002977245021611452 q75: 0.004464050522074103 q90: 0.006034028204157949 max: 0.014522284


In [ ]:
# =============================================================================================================
# ======================================= PREPARE TRADER INPUT ================================================
# =============================================================================================================

for y in YEARS:
    for q in QUARTERS:
        outputs_by_year[y][q]["rv"] = Xq_by_year[y][q]["rv"].reindex(
            outputs_by_year[y][q].index
        )

        outputs_by_year[y][q]["MFIV_24"] = Xq_by_year[y][q]["MFIV_24"].reindex(
            outputs_by_year[y][q].index
        )

In [ ]:
# =============================================================================================================
# ======================================= TSM TRADER ADAPTER ==================================================
# =============================================================================================================
# EXISTING TRADER STILL USES THE TCN NEURAL EXECUTION BRANCH.

for y in YEARS:
    for q in QUARTERS:
        outputs_by_year[y][q]["tcn_action"] = outputs_by_year[y][q]["tsm_action"]
        outputs_by_year[y][q]["tcn_best_score"] = outputs_by_year[y][q]["tsm_best_score"]

In [ ]:
# =============================================================================================================
# ======================================= CACHE FOR WARMUP ====================================================
# =============================================================================================================

CACHE_YEARS = [2019, 2020, 2021]

trader_cache = {
    y: pd.concat(
        [
            outputs_by_year[yy][q]
            for yy in CACHE_YEARS
            if yy <= y
            for q in QUARTERS
        ]
    ).sort_index()
    for y in CACHE_YEARS
}

In [ ]:
# =============================================================================================================
# ================================ TSM_THRESHOLD_TRADER_CALIBRATION ===========================================
# =============================================================================================================

sys.modules.pop("srcB.trader_testing_tcn", None)
from srcB.trader_testing_tcn import testing_execute_trading_strategy


THRESHOLDS = [0.0]

for q in [.25, .50, .75, .90]:
    threshold = min(
        tests[y]["tsm_best_score"].quantile(q)
        for y in YEARS
    )

    THRESHOLDS.append(float(threshold))


THRESHOLDS = sorted(set(THRESHOLDS))

print("TSM trader thresholds:", THRESHOLDS)


rows = []

for threshold in THRESHOLDS:
    for y in YEARS:

        market = Market(
            option_df=df_raw_by_year[y],
            time_df=d_btc_raw_by_year[y],
            starting_cash=100_000,
            im_multiplier=0.15,
            mm_buffer_rate=0.05,
        )

        snapshots, greek_tests = testing_execute_trading_strategy(
            bt_year=y,
            bt_quarter=None,
            model="tcn_mfiv",
            outputs_by_year=outputs_by_year,
            market=market,
            trader_cache=trader_cache,
            run_by="year",
            tcn_score_threshold=threshold
        )

        equity = pd.Series(
            [s["equity"] for s in snapshots],
            index=[s["time"] for s in snapshots]
        )

        returns = equity.pct_change().dropna()

        sharpe = (
            returns.mean()
            / returns.std()
            * np.sqrt(24 * 365)
        )

        rows.append({
            "threshold": threshold,
            "year": y,
            "trades": len(greek_tests),
            "sharpe": sharpe
        })


threshold_trader_check = pd.DataFrame(rows)

print(threshold_trader_check.to_string(index=False))

TSM trader thresholds: [0.0, 0.0007452946738339961, 0.0017103624995797873, 0.0030653050635010004, 0.004852530546486378]
 threshold  year  trades    sharpe
  0.000000  2019     205  0.054521
  0.000000  2020     247  0.650519
  0.000000  2021     260  0.295883
  0.000745  2019     206 -0.027698
  0.000745  2020     216  0.605394
  0.000745  2021     239  0.423263
  0.001710  2019     193  0.114028
  0.001710  2020     138  0.103135
  0.001710  2021     265  0.275008
  0.003065  2019     171  0.312570
  0.003065  2020     138  0.207842
  0.003065  2021     246  0.685955
  0.004853  2019     141  0.388158
  0.004853  2020      68  0.139284
  0.004853  2021     161  0.276156
